# Clean school-level variables, 2020-21 to 2022-23

- **Paper:** (data)
- **Original notebook:** `41_updated_clean_school_related_post.ipynb`
- **Reads:**
  - `outputs/distance_to_colleges.pkl`
  - `outputs/enrollment_by_school.pkl`
  - `data/derived/school_AP_IB_courses.pkl`
  - `data/raw/school_related/20-21, 21-22, 22-23/...`
- **Writes:**
  - `outputs/school_related_post_update.pkl`

> Note: The 2021-22 and 2022-23 Smarter Balanced files are "All Students" subsets (Student Group ID == 1) of the original CAASPP files, made by tools/collect_data.py. This notebook keeps only those rows, so results are the same.


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
    # setup
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import requests
import zipfile
# pip install contextily  # see requirements.txt
import contextily as ctx





In [ ]:
# reading data
distances = pd.read_pickle(f'{OUTPUT_DIR}/distance_to_colleges.pkl')


college_enrollment = pd.read_pickle(f'{OUTPUT_DIR}/enrollment_by_school.pkl')

college_enrollment.columns

college_enrollment = college_enrollment[['DistrictCode',"SchoolCode","CountyName","DistrictName","SchoolName",'Latitude', 'Longitude']]

# math test scores
# sb_test_15_16 = pd.read_csv(f"{DATA_DIR}/15-16/sb_ca2016_1_csv_v3.txt",sep =",",encoding = 'latin-1')
sb_test_20_21 = pd.read_csv(f"{DATA_DIR}/school_related/20-21/sb_ca2021_1_csv_v2.txt",sep ="^",encoding = 'latin-1')
sb_test_20_21["Year"] = "2020-21"
sb_test_21_22 = pd.read_csv(f"{DATA_DIR}/school_related/21-22/sb_ca2022_studentgroup1.txt",sep ="^",encoding = 'latin-1')
sb_test_21_22["Year"] = "2021-22"
sb_test_22_23 = pd.read_csv(f"{DATA_DIR}/school_related/22-23/sb_ca2023_studentgroup1.txt",sep ="^",encoding = 'latin-1')
sb_test_22_23["Year"] = "2022-23"

sb_test = pd.concat([sb_test_20_21, sb_test_21_22,sb_test_22_23], ignore_index=True)


# science test


# absences
absent_file = f"{DATA_DIR}/school_related/absent_2021.txt"



absent_20_21 = pd.read_csv(f"{DATA_DIR}/school_related/20-21/chronicabsenteeism21.txt",sep = '\t',encoding='latin-1')

absent_20_21["Year"] = "2020-21"
absent_21_22 = pd.read_csv(f"{DATA_DIR}/school_related/21-22/chronicabsenteeism22-v3.txt",sep = '\t',encoding='latin-1')
absent_21_22["Year"] = "2021-22"
absent_21_22 = absent_21_22[absent_21_22['DASS']!='All']

absent_22_23 = pd.read_csv(f"{DATA_DIR}/school_related/22-23/chronicabsenteeism23.txt",sep = '\t',encoding='latin-1')
absent_22_23["Year"] = "2022-23"
absent_22_23 = absent_22_23[absent_22_23['DASS']!='All']

absent = pd.concat([absent_20_21,absent_21_22,absent_22_23], ignore_index=True)

# teacher experience


staff_demo = pd.read_csv(f'{DATA_DIR}/school_related/StaffDemo18.txt',sep = '\t',encoding='latin-1')
staff_cred = pd.read_csv(f'{DATA_DIR}/school_related/StaffCred18.txt',sep = '\t',encoding='latin-1')
staff_FTE = pd.read_csv(f'{DATA_DIR}/school_related/StaffSchoolFTE18.txt',sep = '\t',encoding='latin-1')

# free and reduced lunch
lunch_file = f"{DATA_DIR}/school_related/free_reduced_2021.xlsx"
# lunch_15_16 = pd.read_excel(f"{DATA_DIR}/15-16/frpm1516.xls", sheet_name = "FRPM School-Level Data ")
lunch_20_21 = pd.read_excel(f"{DATA_DIR}/school_related/20-21/frpm2021.xlsx", sheet_name = "FRPM School-Level Data ",skiprows = 1)
lunch_20_21["Year"] = "2020-21"

lunch_21_22 = pd.read_excel(f"{DATA_DIR}/school_related/21-22/frpm2122_v2.xlsx", sheet_name = "FRPM School-Level Data ",skiprows = 1)
lunch_21_22["Year"] = "2021-22"

lunch_22_23 = pd.read_excel(f"{DATA_DIR}/school_related/22-23/frpm2223.xlsx", sheet_name = "FRPM School-Level Data",skiprows = 1)
lunch_22_23["Year"] = "2022-23"


lunch = pd.concat([lunch_20_21,lunch_21_22,lunch_22_23], ignore_index=True)
# per pupil spending

"""
https://www.cde.ca.gov/fg/ac/es/essappedata.asp
DNR = Did Not Report; LEA = Local Education Agency; CDS = County District School
"""
spending_20_21 = pd.read_excel(f"{DATA_DIR}/school_related/20-21/essappe2021data.xlsx", skiprows=6,sheet_name="School Level Data")
spending_20_21["Year"] = "2020-21"
spending_21_22 = pd.read_excel(f"{DATA_DIR}/school_related/21-22/essappe2122data.xlsx", skiprows=6,sheet_name="School Level Data")
spending_21_22["Year"] = "2021-22"

spending_22_23 = pd.read_excel(f"{DATA_DIR}/school_related/22-23/essappe2223data.xlsx", skiprows=6,sheet_name="School Level Data")
spending_22_23["Year"] = "2022-23"

spending = pd.concat([spending_20_21,spending_21_22,spending_22_23])
# census tractsUnduplicated Student Poverty – Free or Reduced Price Meals Data 2022–2023
tracts = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/ca_tracts_2022')

# course offerings
courses =pd.read_pickle(f"{DERIVED_DIR}/school_AP_IB_courses.pkl")


In [ ]:
college20 = pd.read_csv(f"{DATA_DIR}/school_related/20-21/cgr12mo21.txt",sep = '\t',encoding='latin-1')
college20["Year"] = "2020-21"
college21 = pd.read_csv(f"{DATA_DIR}/school_related/21-22/cgr12mo22.txt",sep = '\t',encoding='latin-1')
college21["Year"] = "2021-22"

college22 = pd.read_csv(f"{DATA_DIR}/school_related/22-23/cgr12mo23.txt",sep = '\t',encoding='latin-1')
college22["Year"] = "2022-23"

college = pd.concat([college20,college21,college22])
college =college[college['AggregateLevel']=="S"]
# college =college[college['CharterSchool']=="All"]
college = college[college['ReportingCategory']=="TA"]
college = college[college['CompleterType']=="TA"]

# college =college[college['AlternativeSchoolAccountabilityStatus']=="All"].reset_index(drop=True)
college = college[["Year",'SchoolCode','College Going Rate - Total (12 Months)',"Enrolled In College - Total (12 Months)",'High School Completers']]

college.rename(columns = {'College Going Rate - Total (12 Months)':'college_going_rate',
                          "Enrolled In College - Total (12 Months)":'college_going_count',
                          'High School Completers':'hs_completers'},inplace=True)



In [ ]:
courses = courses.rename(columns = {"schoolCode":"SchoolCode"})
courses = courses[['SchoolCode',"IB_Count","AP_Count"]]


In [ ]:
# subsetting to schoolwide data/ taking columns we want
#college_enrollment.columns
# college_enrollment = college_enrollment[['SchoolCode','SchoolName','CountyName','College Going Rate - Total (12 Months)','Latitude',"Longitude",'High School Completers','Enrolled In College - Total (12 Months)']]
# college_enrollment = college_enrollment.rename(columns = {'College Going Rate - Total (12 Months)':'college_going_rate','High School Completers':'hs_completers','Enrolled In College - Total (12 Months)':'enrolled_college'})
absent = absent[absent['Aggregate Level']=='S']
absent = absent[absent['Reporting Category']=='TA']
absent = absent[absent['Charter School'] !='All']
absent.columns
absent = absent[['School Code',"School Name","County Name",'ChronicAbsenteeismRate', 'ChronicAbsenteeismEligibleCumula', 'ChronicAbsenteeismCount',"Year"]]
absent["ChronicAbsenteeismRate"] = pd.to_numeric(absent["ChronicAbsenteeismRate"],errors = 'coerce')
# absent.dropna(inplace = True)

spending['school_expenditures'] = spending['School Expenditures - Federal']+spending['School Expenditures - State & Local']
spending = spending[['School Name','County','school_expenditures',"Year"]]
lunch.columns
lunch = lunch[['School Code','County Name','Percent (%) \nEligible FRPM \n(K-12)','FRPM Count \n(K-12)', 'Enrollment \n(K-12)',"Year"]]
lunch = lunch.rename(columns = {'Percent (%) \nEligible FRPM \n(K-12)':'pct_frpm','FRPM Count \n(K-12)':'frpm_count', 'Enrollment \n(K-12)':'n_students'})

In [ ]:
# smarter balanced assessment cleaning
#sb_test = sb_test[sb_test['Type ID']==7]
sb_test
sb_test['Percentage Standard Met and Above'] = pd.to_numeric(sb_test['Percentage Standard Met and Above'], errors='coerce')

sb_test = sb_test[sb_test['Student Group ID']==1]

sb_test = sb_test[sb_test['Grade']==13]
sb_test['School Code'] = pd.to_numeric(sb_test['School Code'], errors='coerce')


sb_test = sb_test[(sb_test['School Code'] > 0) & (~sb_test['School Code'].isna())]


sb_test_math = sb_test[sb_test['Test ID']==2]
sb_test_math = sb_test_math[['School Code','Percentage Standard Met and Above',"Year"]]

sb_test_math = sb_test_math.rename(columns = {'Percentage Standard Met and Above':'pct_passed_math'})

sb_test_english = sb_test[sb_test['Test ID']==1]
sb_test_english = sb_test_english[['School Code','Percentage Standard Met and Above',"Year"]]

sb_test_english = sb_test_english.rename(columns = {'Percentage Standard Met and Above':'pct_passed_english'})

sb_test = pd.merge(sb_test_math,sb_test_english,on = ['School Code',"Year"])
sb_test= sb_test.dropna()

# sb_test = sb_test.groupby('School Code')['sb_mean_score'].mean().reset_index()




In [ ]:
# teacher experience cleaning
staff_demo = staff_demo.drop(columns = ["AcademicYear","FileCreated","FTE Teaching","FTE administrative","FTE PupilServices"])

#staff_demo = pd.merge(staff_demo,staff_cred,on = ['RecID','AcademicYear'],how = 'left')
staff_demo = pd.merge(staff_demo,staff_FTE,on = ['RecID'],how = 'left')

"""
D = Doctorate
S = Special
V = Master's degree plus 30 or more semester hours
M = Master's degree
U = Fifth year within bachelor's degree
Y = Fifth year induction
F = Fifth year
C = Baccalaureate plus 30 or more semester hours
B = Baccalaureate
A = Associate degree
N = Not reported
"""
staff_demo['BeyondBachelors'] = staff_demo['EducationLevel'].isin(['D', 'S', 'V', 'M'])
bins =[0,3,10,float('inf')]

staff_demo['experience_level'] = pd.cut(staff_demo['YearsTeaching'],bins = bins,labels = [1,2,3],right=False)
staff_demo = pd.get_dummies(staff_demo, columns=['experience_level'], prefix='TeachingExp')


staff_demo= staff_demo[staff_demo['StaffType']=='T']
total_teachers = staff_demo.groupby('SchoolCode').size()
staff_demo["years_teaching_std"] = staff_demo['YearsTeaching']
teacher_experience =staff_demo.groupby(['SchoolCode','CountyName_y']).agg({

    'YearsTeaching': 'mean',
    'years_teaching_std':'std'
   # ,'TeachingExp_1':'mean',
   # 'TeachingExp_2':'mean',
   # 'TeachingExp_3':'mean'
}).reset_index()

# teacher_experience['n_teachers'] = total_teachers.values
# other data already in schoolwide form
teacher_experience.drop(columns = 'CountyName_y',inplace = True)

In [ ]:
college_enrollment['SchoolCode'] = college_enrollment['SchoolCode'].astype('float64')
courses['SchoolCode'] = courses['SchoolCode'].astype('float64')


merged_df = pd.merge(college_enrollment,teacher_experience,on = ["SchoolCode"],how ='inner')
merged_df = pd.merge(merged_df,courses,on =  ['SchoolCode'],how = 'left')



merged_df = pd.merge(merged_df,distances,on = ['SchoolName',"CountyName"],how = 'inner')

merged_df = pd.merge(merged_df, spending, left_on= ["SchoolName",'CountyName'],right_on = ["School Name","County"],how = 'inner')
merged_df = merged_df.drop(columns = ['School Name','County'])


merged_df = pd.merge(merged_df,absent,left_on=["SchoolName","CountyName","Year"],right_on =['School Name',"County Name","Year"],how ='left')

merged_df = merged_df.drop(columns = 'School Code')


merged_df = pd.merge(merged_df,lunch,left_on=["SchoolCode","Year"],right_on =['School Code',"Year"],how ='left')
merged_df = merged_df.drop(columns = 'School Code')


merged_df = pd.merge(merged_df,sb_test,left_on=["SchoolCode","Year"],right_on =['School Code',"Year"],how ='left')
merged_df = merged_df.drop(columns = ['School Code',"County Name_x","County Name_y"])


merged_df = pd.merge(merged_df,college,on = ['SchoolCode',"Year"])
merged_df.columns.values
with open(f'{OUTPUT_DIR}/school_related_post_updated.pkl','wb') as f:
  pickle.dump(merged_df,f)


In [ ]:
relevant  = merged_df.drop(columns = ["SchoolCode", "SchoolName", "CountyName", "Latitude", "Longitude", "DistrictCode",'DistrictName','closest_UC','closest_CSU',"frpm_count","ChronicAbsenteeismCount", "ChronicAbsenteeismEligibleCumula"])


for col in relevant.columns:
    relevant[col] = pd.to_numeric(relevant[col], errors='coerce')

with open(f'{OUTPUT_DIR}/school_related_post_update.pkl','wb') as f:
  pickle.dump(merged_df,f)



In [ ]:
sns.pairplot(relevant)
plt.show()

In [ ]:
cor_mat = relevant.corr(numeric_only = True)
matrix = np.triu(cor_mat)

plt.figure(figsize=(12, 8))
sns.heatmap(cor_mat, annot=True, cmap='coolwarm', fmt=".2f",mask=matrix)
plt.title('Correlation Matrix')
plt.show()

In [ ]:

# # Set marker properties
# marker_size = 50
# marker_color = 'black'
# marker_edge_color = 'white'
# marker_edge_width = 0.5

# # FRPM Percent vs. College Going Rate
# plt.figure(figsize=(8, 6))
# plt.scatter(relevant['college_going_rate'],
#             relevant['pct_frpm'],
#             s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
# plt.title('College Going Rate vs. FRPM Percent')
# plt.xlabel('College Going Rate')
# plt.ylabel('FRPM Percent')
# plt.savefig(f'{OUTPUT_DIR}/plots/frpm_vs_college.png', dpi=500)
# plt.close()

# # SB Mean Score vs. College Going Rate
# plt.figure(figsize=(8, 6))
# plt.scatter(relevant['college_going_rate'],
#             relevant['pct_passed_math'],
#             s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
# plt.title('College Going Rate vs. Smarter Balanced Assessment Mean Score')
# plt.xlabel('College Going Rate')
# plt.ylabel('SB Mean Score')
# plt.savefig(f'{OUTPUT_DIR}/plots/sb_mean_score_vs_college.png', dpi=500)
# plt.close()

# # CAST Mean Score vs. College Going Rate
# plt.figure(figsize=(8, 6))
# plt.scatter(relevant['college_going_rate'],
#             relevant['pct_passed_science'],
#             s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
# plt.title('College Going Rate vs. CAST Mean Score')
# plt.xlabel('College Going Rate')
# plt.ylabel('CAST Mean Score')
# plt.savefig(f'{OUTPUT_DIR}/plots/cast_mean_score_vs_college.png', dpi=500)
# plt.close()

# # Chronic Absenteeism Rate vs. College Going Rate
# plt.figure(figsize=(8, 6))
# plt.scatter(relevant['college_going_rate'],
#             relevant['ChronicAbsenteeismRate'],
#             s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
# plt.title('College Going Rate vs. Chronic Absenteeism Rate')
# plt.xlabel('College Going Rate')
# plt.ylabel('Chronic Absenteeism Rate')
# plt.savefig(f'{OUTPUT_DIR}/plots/chronic_absenteeism_rate_vs_college.png', dpi=500)
# plt.close()


In [ ]:
merged_df = merged_df[['Latitude',"Longitude",'college_going_rate']]
located_schools = gpd.GeoDataFrame(merged_df,geometry=gpd.points_from_xy(merged_df['Longitude'], merged_df['Latitude']), crs='EPSG:4326')

located_schools['college_going_rate'] = pd.to_numeric(located_schools['college_going_rate'], errors='coerce')
plt.figure(figsize=(8, 6))

ax = located_schools.plot(column='college_going_rate', cmap='magma', legend=False, markersize=5, figsize=(10, 6))


#  basemap
ctx.add_basemap(ax=ax, crs=located_schools.crs.to_string(), source=ctx.providers.CartoDB.Positron)

# colorbar
cbar = plt.colorbar(ax.collections[0], ax=ax, label='College Going Rate (%)')
ticks = np.linspace(0,100, num=5)
cbar.set_ticks(ticks)
plt.title("College going rate in California Public High Schools")
ax.tick_params(axis='both', which='both', length=0, labelsize=0)
plt.show()
# plt.savefig(f'{OUTPUT_DIR}/plots/college_going_rate_map.png', dpi=500)
# plt.close()
